# Week 7 Coding: Solutions

Three tasks connected to the Week 7 problem set: **C1** ↔ P8, **C2** ↔ P7, **C3** ↔ the transient of Video 5 (**optional**).

**Access through Moodle.** Open the activity **“L07 - Post-class - Jupyter Notebook Link”** for the student exercise. On your first visit, click **“Get a copy of the assignment”** if prompted to create your personal copy in ETH JupyterHub. If prompted, select **“Start My Server”** and wait for JupyterLab to open. Return to the notebook through the same Moodle activity on later visits. The completed reference notebook is provided through a **separate Moodle activity**.

**Reference solutions.** This notebook contains the completed code. Read the corresponding question in the problem set, predict its result, then run the cells in order with **Shift+Enter**. The check cells report success or raise an `AssertionError` if a check fails.

**If something goes wrong:** In JupyterLab, use **Kernel → Restart Kernel**, then run the cells again from the top in order.


## C1: The frequency response on a grid *(↔ P8)*

For $H(z) = \dfrac{b_0 z^M + \dots + b_M}{a_0 z^N + \dots + a_N}$, the frequency response is $H(\Omega) = H(z)\rvert_{z = e^{j\Omega}}$: evaluate the numerator and the denominator polynomials at the points $e^{j\Omega}$ and divide.

> **Note on `np.polyval(p, x)`:**
> * **Polynomial ordering:** `np.polyval(p, x)` expects coefficients `p` ordered from highest power to lowest power:
>   $$p(x) = p[0]x^N + p[1]x^{N-1} + \dots + p[N]$$
> * **Complex & array inputs:** If `x` is an array of complex points on the unit circle ($z = e^{j\Omega}$), `np.polyval` evaluates the polynomial element-wise at each point automatically.

a) Complete `freq_response`. The check cell compares it with the closed form $\lvert H(\Omega)\rvert = 1/\sqrt{1 - 2a\cos\Omega + a^2}$ of P8 b) and with the values $5$, $0.781$, $0.556$ of the pre-class example.

b) Run the plot cell: the Bode plots of $z/(z - 0.8)$ and $z/(z + 0.8)$, the high-pass of P8 c).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def freq_response(num, den, Omega):
    """Evaluates transfer function H(z) = num(z) / den(z) on the unit circle z = exp(j * Omega).
    Inputs:
      - num, den: Coefficient arrays in descending powers of z.
      - Omega: Digital frequencies in radians array.
    Hint:
      - Use np.polyval(..., ...) for the numerator and denominator.
    """
    z = np.exp(1j * Omega)
    return np.polyval(num, z) / np.polyval(den, z)

Omega = np.linspace(0, np.pi, 500)
H = freq_response([1, 0], [1, -0.8], Omega)    # H(z) = z / (z - 0.8)
print(np.abs(freq_response([1, 0], [1, -0.8], np.array([0, np.pi/2, np.pi]))))

In [ ]:
a = 0.8
assert np.allclose(np.abs(H), 1 / np.sqrt(1 - 2*a*np.cos(Omega) + a**2))                 # P8 b)
vals = np.abs(freq_response([1, 0], [1, -0.8], np.array([0, np.pi/2, np.pi])))
assert np.allclose(vals, [5, 0.781, 0.556], atol=1e-3)                                    # the pre-class example
assert np.allclose(np.round(20*np.log10(vals), 1), [14.0, -2.1, -5.1])
print("C1 passed: the grid evaluation reproduces the closed form and the values 5, 0.781, 0.556.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
for a, lab in ((0.8, "$z/(z-0.8)$"), (-0.8, "$z/(z+0.8)$")):
    Ha = freq_response([1, 0], [1, -a], Omega)
    ax1.plot(Omega, 20*np.log10(np.abs(Ha)), label=lab)
    ax2.plot(Omega, np.angle(Ha), label=lab)
ax1.set_ylabel("magnitude (dB)"); ax2.set_ylabel("phase (rad)")
for ax in (ax1, ax2):
    ax.set_xlabel(r"$\Omega$"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()

## C2: Linear phase of a symmetric FIR system *(↔ P7)*

For $h = \{1, 2, 1\}$, i.e. $H(z) = \dfrac{z^2 + 2z + 1}{z^2}$, the phase is computed on a grid and converted to a delay in samples, $-\angle H(\Omega)/\Omega$.

> **Notes:**
> * The `freq_response` function was defined in **Coding Task 1**.
> * You can use `np.angle(list)`, which computes the phase angle (in radians) for each complex value in an array or list.

Complete the delay line and run the check cell. Then run the last cell: it filters $\cos(\tfrac{\pi}{2}n) + \cos(\tfrac{\pi}{8}n)$ and plots the original (unshifted) input alongside the filtered output. The two frequency components are amplified by different factors, so the common one-sample delay is established by the phase calculation and check cell, not by a direct time-shift comparison of the plotted curves. The first two output samples also reflect the causal filter startup with missing earlier input samples treated as zero.

In [ ]:
Om = np.linspace(0.05, 3.0, 300)                 # avoid Omega = 0 and the zero at Omega = pi
Hs = freq_response([1, 2, 1], [1, 0, 0], Om)
delay = -np.angle(Hs) / Om
print("delay range:", delay.min(), delay.max())

In [ ]:
assert np.allclose(delay, 1.0)                                                 # one sample at every frequency
assert np.isclose(freq_response([1, 2, 1], [1, 0, 0], np.array([np.pi/2]))[0], -2j)  # P7: 2 e^{-j pi/2}
print("C2 passed: the symmetric system delays every frequency by one sample.")

In [ ]:
n = np.arange(40)
u = np.cos(np.pi/2 * n) + np.cos(np.pi/8 * n)
y = np.convolve(u, [1, 2, 1])[:len(n)]
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.stem(n, u, linefmt="C7-", markerfmt="C7o", basefmt="k-", label="input")
ax.plot(n, y, "C0o-", ms=3, label="output")
ax.set_xlabel("$n$"); ax.legend(fontsize=8); plt.tight_layout()
# The two components are scaled differently (|H| = 2 and about 3.85) but both are delayed by one sample.

## C3: The transient of the first-order system *(↔ Video 5)* (optional)

The system $y[n] = 0.8\,y[n-1] + u[n]$, $H(z) = z/(z - 0.8)$, is driven by $u[n] = \cos(\tfrac{\pi}{2}n)\,s[n]$, applied from $n = 0$, with $y[-1] = 0$.

a) Complete the recursion and run the check cell: the difference between the output and the sinusoid response $\lvert H(\tfrac{\pi}{2})\rvert\cos\bigl(\tfrac{\pi}{2}n + \angle H(\tfrac{\pi}{2})\bigr)$ is $c\,0.8^n$.

b) Find $c$ from $y[0] = 1$ by hand, and explain in one sentence why the difference decays like $0.8^n$.

In [ ]:
N = 30
n = np.arange(N)
u = np.cos(np.pi/2 * n)
y = np.zeros(N)
for k in range(N):
    y[k] = (0.8 * y[k-1] if k > 0 else 0.0) + u[k]

Hq = freq_response([1, 0], [1, -0.8], np.array([np.pi/2]))[0]
y_ss = np.abs(Hq) * np.cos(np.pi/2 * n + np.angle(Hq))  # the sinusoid response
d = y - y_ss
print("d[n] / 0.8^n for n = 0..5:", np.round(d[:6] / 0.8**n[:6], 4))

In [ ]:
c = d[0]
assert np.allclose(d, c * 0.8**n)                       # the difference is a single mode
assert abs(c - 0.39) < 0.005                            # c = 1 - 0.781 cos(-0.675) = 0.39
print(f"C3 passed: the transient is c 0.8^n with c = {c:.3f}.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.stem(n, y, basefmt="k-", label="$y[n]$")
ax.plot(n, y_ss, "C7--", label="sinusoid response")
ax.set_xlabel("$n$"); ax.legend(fontsize=8); plt.tight_layout()

### Solution for Subtask b)

**1. Finding $c$ by hand:**

* **Calculate $y[0]$:**
  $$y[0] = 0.8 \cdot y[-1] + \cos(0) = 0 + 1 = 1$$

* **Calculate $y_{ss}[0]$:**
  Evaluate $H(z)$ at $z = e^{j\pi/2} = j$:
  $$H(j) = \frac{j}{j - 0.8} = \frac{1 - 0.8j}{1.64}$$
  
  Taking the real part gives $y_{ss}[0]$:
  $$y_{ss}[0] = \text{Re}\{H(j)\} = \frac{1}{1.64} \approx 0.610$$

* **Calculate $c$:**
  $$c = y[0] - y_{ss}[0] = 1 - 0.610 = 0.390$$

**2. Explanation:**

The difference decays as $0.8^n$ because $z = 0.8$ is the system's pole, which governs the rate at which the natural transient response dies out.